<p><font size="6" color='grey'> <b>

Generative KI. Verstehen. Anwenden. Gestalten.
</b></font> </br></p>

<p><font size="5" color='grey'> <b>
Middleware
</b></font> </br></p>

---

In [ ]:
#@title 🔧 Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/GenAI.git#subdirectory=04_modul

from importlib import metadata as importlib_metadata

# ── Projekt-Utilities ─────────────────────────────────────────────────────────
from genai_lib.utilities import (
    check_environment,
    get_ipinfo,
    setup_api_keys,
    mprint,
    install_packages,
    mermaid,
    get_model_profile,
    extract_thinking,
    load_prompt
)
setup_api_keys(['OPENAI_API_KEY'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

# Versionen sichtbar machen: Middleware-APIs ändern sich zwischen LangChain-Releases.
print("
Paketversionen:")
for package_name in ["langchain", "langchain-core", "langchain-openai", "langgraph"]:
    try:
        version = importlib_metadata.version(package_name)
    except importlib_metadata.PackageNotFoundError:
        version = "nicht installiert"
    print(f"  {package_name}: {version}")

# 1 | Was ist Middleware?
---

**Middleware** ist eine *Zwischenschicht*, die sich zwischen einen **Agenten** und die **Ausführung seiner Tools** schaltet. Sie ermöglicht es, Kontrollmechanismen einzufügen, ohne den Agenten oder die Tools zu verändern.

**Zentrale Aufgaben von Middleware:**

* **Vorverarbeitung (Preprocessing):** Bereinigung von Eingabedaten, Strukturierung von Abfragen oder Ergänzung durch Kontext
* **Kontrolle & Sicherheit:** Implementierung von Content-Filtern, PII-Anonymisierung (Personally Identifiable Information, personenbezogene Daten) und Validierung von Eingabeparametern
* **Nachbearbeitung (Postprocessing):** Formatierung der Rohdaten, Kürzung von Texten oder Anreicherung der Antwort mit Metadaten
* **Instrumentierung & Monitoring:** Systematisches Logging, Performance-Tracking sowie Debugging des gesamten Workflows

Das Konzept ist vergleichbar mit Middleware in Webframeworks (z.B. Express.js, Django): Anfragen durchlaufen eine Kette von Prüfungen, bevor sie ausgeführt werden.

In [ ]:
#@markdown   <p><font size="4" color='green'>  Middleware-Prozess</font> </br></p>

diagram = """
%%{init: {'theme':'light'}}%%
flowchart TD
    %% Definition der Farb-Klassen
    classDef controlStyle fill:#e1f5fe,stroke:#039be5,stroke-width:2px;
    classDef defaultStyle fill:#f4f4f5,stroke:#71717a,stroke-width:1px;

    %% Knoten-Definitionen
    A["`User Input`"]:::defaultStyle
    B["`<b>before_model</b><br/>Was geht rein?`"]:::controlStyle
    C["`<b>MODEL</b><br/>LLM denkt nach`"]:::defaultStyle
    D["`<b>after_model</b><br/>Was kam raus?`"]:::controlStyle
    E{"`Tool-Aufruf?`"}:::defaultStyle
    F["`Fertig`"]:::defaultStyle
    G["`<b>wrap_tool_call</b><br/>Welches Tool? Mit welchen Args?`"]:::controlStyle

    %% Ablauf-Verbindungen
    A --> B
    B --> C
    C --> D
    D --> E

    E -->|Nein| F
    E -->|Ja| G

    G --> B
"""
mermaid(diagram, width=500)

<p><font color='black' size="5">
Zwei Arten von Middleware
</font></p>

LangChain Version 1.0+ bietet zwei Mechanismen:

**1. Decorator-Hooks** (low-level, eigene Logik):


Ein Decorator-Hook erweitert das Verhalten einer Funktion automatisch an definierten Einhängepunkten (z. B. vor, nach oder während der Ausführung).    

| Hook | Wann? | Typische Aufgabe |
|------|-------|-----------------|
| `@before_model` | **Vor** dem LLM-Aufruf | Logging, Input-Validierung, Kontext prüfen |
| `@after_model` | **Nach** dem LLM-Aufruf | Output-Logging, Entscheidung prüfen (Tool vs. Antwort) |
| `@wrap_tool_call` | **Um** die Tool-Ausführung | Tool-Logging, Ergebnisse überwachen, Timing |



**2. Prebuilt Middleware** (high-level, fertige Bausteine):      
Prebuilt Middleware ist fertige Middleware, die standardisierte Funktionen wie Logging, Authentifizierung oder Validierung in einen Workflow integriert.

| Middleware                     | Zweck                                                                                                                               |
| ------------------------------ | ----------------------------------------------------------------------------------------------------------------------------------- |
| HumanInTheLoopMiddleware       | Unterbricht die Ausführung vor kritischen Schritten, damit ein Mensch Freigaben erteilen oder Eingaben korrigieren kann.            |
| SummarizationMiddleware        | Verdichtet lange Gesprächs- oder Nachrichtenverläufe, wenn der Kontext zu groß wird.                                                |
| ToolRetryMiddleware            | Wiederholt fehlgeschlagene Tool-Aufrufe automatisch mit einer Retry-Strategie, oft mit Backoff.                                     |
| ModelRetryMiddleware           | Wiederholt fehlgeschlagene Modellaufrufe automatisch, etwa bei temporären API-Fehlern.                                              |
| ModelCallLimitMiddleware       | Begrenzt die Anzahl der Modellaufrufe innerhalb eines Agentenlaufs oder einer Session.                                              |
| ToolCallLimitMiddleware        | Begrenzt die Anzahl der Tool-Aufrufe insgesamt oder pro Tool.                                                                       |
| ModelFallbackMiddleware        | Wechselt bei Ausfall des Primärmodells auf ein Ersatzmodell.                                                                        |
| PIIMiddleware                  | Erkannt und behandelt personenbezogene Daten wie E-Mail-Adressen, Telefonnummern oder Kreditkartendaten.                            |
| ContextEditingMiddleware       | Bearbeitet den Verlauf nachträglich, zum Beispiel durch Kürzen oder Entfernen alter Tool-Nutzung, um den Kontext schlank zu halten. |
| LLMToolEmulator                | Simuliert Tool-Ausführungen mithilfe eines LLM, zum Beispiel für Tests ohne echte Nebenwirkungen.                                   |
| LLMToolSelectorMiddleware      | Lässt ein LLM vorfiltern, welche Tools für einen Task relevant sind.                                                                |
| TodoListMiddleware             | Unterstützt Aufgabenplanung und Fortschrittsverfolgung in agentischen Workflows.                                                    |
| ShellToolMiddleware            | Stellt eine persistente Shell-Session als ausführbares Tool bereit.                                                                 |
| FilesystemFileSearchMiddleware | Ermöglicht Datei-, Glob- oder Grep-Suchen im Dateisystem.                                                                           |

Alle importierbar aus: `from langchain.agents.middleware import <Klasse>`

# 2 | Setup: Modell und Tools
---

Bevor wir Middleware einsetzen können, Definition eines LLM und einfache Tools, die der Agent verwenden kann.

In [ ]:
# ── LangChain ─────────────────────────────────────────────────────────────────
from langchain.agents import AgentState, create_agent
from langchain.agents.middleware import (
    HumanInTheLoopMiddleware,
    ModelRetryMiddleware,
    SummarizationMiddleware,
    ToolRetryMiddleware,
    after_model,
    before_model,
    wrap_tool_call,
)
from langchain.chat_models import init_chat_model
from langchain.tools.tool_node import ToolCallRequest
from langchain_core.tools import tool

# ── LangGraph ─────────────────────────────────────────────────────────────────
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

# ── Projekt-Utilities ─────────────────────────────────────────────────────────
from genai_lib.model_config import BASELINE

# LLM initialisieren
llm = init_chat_model(BASELINE)

In [ ]:
# Tools definieren
@tool
def rechner(ausdruck: str) -> str:
    """Führt eine mathematische Berechnung durch.

    Args:
        ausdruck: Mathematischer Ausdruck, z.B. '42 * 17'

    Returns:
        Ergebnis der Berechnung
    """
    try:
        if any(x in ausdruck for x in ['import', 'exec', '__']):
            return "Unsichere Operation"
        result = eval(ausdruck)
        return f"{ausdruck} = {result}"
    except Exception:
        return "Berechnungsfehler"


@tool
def datei_lesen(dateiname: str) -> str:
    """Liest den Inhalt einer Datei.

    Args:
        dateiname: Name der zu lesenden Datei

    Returns:
        Inhalt der Datei als Text
    """
    return f"Inhalt von '{dateiname}': Dies ist ein Beispieltext."


@tool
def datei_schreiben(dateiname: str, inhalt: str) -> str:
    """Erstellt oder überschreibt eine Datei mit dem angegebenen Inhalt.

    Args:
        dateiname: Name der zu schreibenden Datei
        inhalt: Text-Inhalt der Datei

    Returns:
        Bestätigung der Schreiboperation
    """
    return f"Datei '{dateiname}' erfolgreich geschrieben."


tools = [rechner, datei_lesen, datei_schreiben]
print(f"Tools: {[t.name for t in tools]}")

# 3 | before_model / after_model / wrap_tool_call
---

Die drei Decorator-Hooks bilden das Kernstück der Middleware in LangChain 1.0+. Sie erlauben es, den gesamten Agent-Zyklus zu beobachten und zu steuern:

- `@before_model` – wird **vor** jedem LLM-Aufruf ausgeführt. Parameter: `state` (AgentState) und `runtime`.
- `@after_model` – wird **nach** jedem LLM-Aufruf ausgeführt. Parameter: `state` (AgentState) und `runtime`.
- `@wrap_tool_call` – wird **um** jeden Tool-Aufruf gewickelt. Parameter: `request` (ToolCallRequest) und `handler`.

Jeder Hook gibt `None` zurück (passiert den State unverändert durch) oder kann den State modifizieren.

<p><font color='black' size="5">
Middleware definieren
</font></p>

In [ ]:
import re


def redact_for_log(value, max_chars: int = 80) -> str:
    """Maskiert einfache PII-Muster für Demo-Logs."""
    text = str(value)
    text = re.sub(r"[\w\.-]+@[\w\.-]+\.\w+", "[EMAIL]", text)
    text = re.sub(r"\b(?:\+?\d[\d .\-/]{6,}\d)\b", "[TELEFON]", text)
    if len(text) > max_chars:
        return text[:max_chars] + "..."
    return text


@before_model
def log_before(state: AgentState, runtime):
    """Wird VOR jedem LLM-Aufruf ausgeführt."""
    print(f"\nModel wird aufgerufen mit {len(state['messages'])} Nachrichten")
    for msg in state["messages"][-2:]:
        role = msg.type if hasattr(msg, "type") else "unknown"
        content = redact_for_log(msg.content)
        print(f"   [{role}]: {content}")
    return None


@after_model
def log_after(state: AgentState, runtime):
    """Wird NACH jedem LLM-Aufruf ausgeführt."""
    msg = state["messages"][-1]
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        tool_names = [tc["name"] for tc in msg.tool_calls]
        print(f"Tool-Aufruf geplant: {tool_names}")
    else:
        print("Antwort generiert")
    return None


@wrap_tool_call
def log_tool(request: ToolCallRequest, handler):
    """Wird UM jeden Tool-Aufruf gewickelt."""
    tool_name = request.tool_call["name"]
    tool_args = redact_for_log(request.tool_call.get("args", {}), max_chars=120)
    print(f"Tool wird ausgeführt: {tool_name}({tool_args})")
    result = handler(request)
    print(f"Tool-Ergebnis: {redact_for_log(result.content, max_chars=100)}")
    return result

<p><font color='black' size="5">
Agent mit Middleware erstellen und testen
</font></p>

In [ ]:
# Agent mit allen drei Middleware-Hooks erstellen
system_prompt_logging = "Du bist ein hilfreicher Assistent. Antworte auf Deutsch."

agent = create_agent(
    model=llm,
    tools=[rechner, datei_lesen],
    system_prompt=system_prompt_logging,
    middleware=[log_before, log_after, log_tool]
)

print("✅ Agent mit before_model / after_model / wrap_tool_call erstellt")

In [ ]:
# Test: Anfrage die ein Tool verwendet
result = agent.invoke({
    "messages": [{"role": "user", "content": "Was ist 2847 * 1923?"}]
})

print()
mprint("### 📣 Antwort")
mprint("---")
mprint(result["messages"][-1].content)

In [ ]:
# Test: Anfrage ohne Tool (direkte Antwort)
result = agent.invoke({
    "messages": [{"role": "user", "content": "Was ist Middleware?"}]
})

print()
mprint("### 📣 Antwort")
mprint("---")
mprint(result["messages"][-1].content)

<p><font color='darkblue' size="4">
ℹ️ <b>Was passiert hier?</b>
</font></p>

In der Konsole sind die Middleware-Logs sichtbar:

1. **`Model wird aufgerufen`** – `before_model` zeigt die eingehenden Nachrichten, gekürzt und mit einfacher Redaction
2. **`Tool-Aufruf geplant`** – `after_model` erkennt, dass das LLM ein Tool aufrufen möchte
3. **`Tool wird ausgeführt`** – `wrap_tool_call` loggt den Tool-Aufruf und das Ergebnis
4. **`Model wird aufgerufen`** – Der Agent macht eine zweite Runde mit dem Tool-Ergebnis
5. **`Antwort generiert`** – `after_model` erkennt, dass jetzt eine finale Antwort kommt

# 4 | HumanInTheLoopMiddleware
---

Nicht jedes Tool sollte ohne Rückfrage ausgeführt werden. Besonders bei **schreibenden Operationen** ist eine menschliche Bestätigung sinnvoll. Die `HumanInTheLoopMiddleware` unterbricht den Agenten vor der Ausführung bestimmter Tools und fragt den Nutzer um Erlaubnis.

| Szenario | Beispiel-Tool | Warum user-in-the-Loop? |
|----------|---------------|--------------------------|
| Dateien ändern | `datei_schreiben`, `datei_loeschen` | Datenverlust vermeiden |
| Externe Kommunikation | `send_email`, `post_message` | Ungewollte Nachrichten verhindern |
| Finanztransaktionen | `transfer_money`, `place_order` | Fehlerhafte Buchungen vermeiden |
| Systemänderungen | `deploy`, `restart_service` | Ausfälle vermeiden |

**Prinzip:** Der Agent darf *lesen* und *denken* ohne Einschränkung – aber *handeln* nur mit Zustimmung.

In [ ]:
#@markdown   <p><font size="4" color='green'>  Human-in-the-Loop Ablauf</font> </br></p>

diagram = """
%%{init: {'theme':'forest'}}%%
flowchart TD
    A[User Input] --> B[Agent / LLM]
    B --> C{Tool-Aufruf?}

    C -->|Nein| D[Direkte Antwort]
    C -->|Ja| E{Tool in<br/>Sperrliste?}

    E -->|Nein| F[Tool wird<br/>automatisch ausgeführt]
    E -->|Ja| G[⏸️ PAUSE<br/>Nutzer wird gefragt]

    G --> H{Nutzer<br/>erlaubt?}
    H -->|✅ Ja ... | F
    H -->|❌  Nein ... | I[Tool-Aufruf<br/>abgelehnt]

    F --> J[Ergebnis → Agent]
    I --> J
    J --> B

    style G fill:#fff3cd,stroke:#ffc107,stroke-width:2px
    style H fill:#fff3cd,stroke:#ffc107,stroke-width:2px
    style F fill:#d4edda,stroke:#28a745
    style I fill:#f8d7da,stroke:#dc3545
"""
mermaid(diagram, width=400)

<p><font color='black' size="5">
Agent mit HumanInTheLoopMiddleware erstellen
</font></p>

Für die `HumanInTheLoopMiddleware` wird ein **Checkpointer** benötigt (`InMemorySaver`), damit der Agent nach der Unterbrechung an derselben Stelle fortfahren kann. Der Parameter `interrupt_on` definiert, welche Tools eine Genehmigung erfordern.

In [ ]:
# HITL-Middleware: datei_schreiben erfordert Bestätigung
hitl = HumanInTheLoopMiddleware(
    interrupt_on={"datei_schreiben": True}
)

system_prompt_hitl = "Du bist ein hilfreicher Datei-Assistent. Antworte auf Deutsch."

# Agent MIT Checkpointer (nötig für Interrupt/Resume)
agent_hitl = create_agent(
    model=llm,
    tools=tools,
    system_prompt=system_prompt_hitl,
    middleware=[hitl],
    checkpointer=InMemorySaver()
)

config = {"configurable": {"thread_id": "demo-hitl"}}
print("Agent mit HumanInTheLoopMiddleware erstellt")

In [ ]:
def handle_hitl_result(result, agent, config):
    """Behandelt HITL-Interrupts und funktioniert mit dict- und object-artigen Ergebnissen."""
    interrupts = result.get("__interrupt__") if isinstance(result, dict) else getattr(result, "interrupts", None)
    if not interrupts:
        print("Kein Interrupt nötig - Agent hat direkt geantwortet:\n")
        print(result["messages"][-1].content)
        return result

    print("Agent unterbrochen - wartet auf Genehmigung\n")
    antwort = input("Tool ausführen? (ja/nein): ")
    decision_type = "approve" if antwort.strip().lower() in ("ja", "j", "yes", "y") else "reject"
    resumed = agent.invoke(
        Command(resume={"decisions": [{"type": decision_type}]}),
        config=config
    )
    print()
    print(resumed["messages"][-1].content)
    return resumed


# Lesen läuft ohne Unterbrechung
result = agent_hitl.invoke(
    {"messages": [{"role": "user", "content": "Lies bitte die Datei bericht.txt"}]},
    config=config
)

result = handle_hitl_result(result, agent_hitl, config)

In [ ]:
# Test: Schreiben wird unterbrochen - Nutzer entscheidet
result = agent_hitl.invoke(
    {"messages": [{"role": "user", "content": "Schreibe 'Hallo Welt' in die Datei test.txt"}]},
    config=config
)

result = handle_hitl_result(result, agent_hitl, config)

<p><font color='darkblue' size="4">
ℹ️ <b>Wie funktioniert der Interrupt?</b>
</font></p>

1. Der Agent plant den Aufruf von `datei_schreiben`
2. Die `HumanInTheLoopMiddleware` erkennt das Tool in der `interrupt_on`-Liste
3. Der Agent wird **pausiert** – der Zustand wird im `InMemorySaver` gespeichert
4. Der Nutzer gibt seine Entscheidung ein (`approve` oder `reject`)
5. `Command(resume=...)` setzt den Agenten mit der Entscheidung fort

**Wichtig:** Ohne `checkpointer=InMemorySaver()` kann der Agent nach dem Interrupt nicht fortgesetzt werden!

# 5 | SummarizationMiddleware
---

Die `SummarizationMiddleware` löst ein zentrales Problem langer Konversationen: **Token-Überlauf**. Wenn eine Session zu lang wird, fasst sie den bisherigen Verlauf automatisch zusammen und hält dabei die neuesten Nachrichten im Detail.

**Parameter:**

| Parameter | Typ | Beschreibung |
|-----------|-----|-------------|
| `model` | `str` oder `BaseChatModel` | Modell für die Zusammenfassung |
| `trigger` | `ContextSize` | Token-/Nachrichtengrenze, ab der zusammengefasst wird |
| `keep` | `ContextSize` | Wie viel Kontext erhalten bleiben soll |
| `token_counter` | `callable` | Optionale Funktion zum Token-Zählen |
| `summary_prompt` | `str` | Optionales Template mit `{messages}` Platzhalter |

In [ ]:
# Agent mit automatischer Kontext-Zusammenfassung
system_prompt_summary = "Du bist ein hilfreicher Assistent. Antworte auf Deutsch."

agent_summary = create_agent(
    model=llm,
    tools=[rechner, datei_lesen],
    system_prompt=system_prompt_summary,
    middleware=[SummarizationMiddleware(model=llm)]
)

print("✅ Agent mit SummarizationMiddleware erstellt")

**Alternative: OpenAI Server-Side Compaction** *(neu: langchain-openai 1.1.10, Feb 2026)*

Für OpenAI-Modelle gibt es seit Februar 2026 eine einfachere Alternative zur `SummarizationMiddleware`:
OpenAI komprimiert die Konversationshistorie **serverseitig** – kein extra Middleware-Layer nötig.

| Ansatz | Wie | Provider | Wann verwenden? |
|--------|-----|----------|-----------------|
| `SummarizationMiddleware` | Client: LLM fasst lokal zusammen | Alle Provider | Provider-unabhängig, mehr Kontrolle |
| `context_management` (neu) | Server: OpenAI komprimiert | Nur OpenAI | Einfachste Lösung für OpenAI-Apps |

In [ ]:
# Alternative zu SummarizationMiddleware für OpenAI-Modelle (langchain-openai 1.1.10)
llm_mit_compaction = init_chat_model(
    BASELINE,
    context_management=[{"type": "compaction", "compact_threshold": 10_000}]
    # Komprimiert automatisch, wenn ~10.000 Tokens überschritten werden
    # Kein extra Middleware-Layer nötig – alles passiert auf OpenAI-Seite
)

system_prompt_compaction = "Du bist ein hilfreicher Assistent. Antworte auf Deutsch."

agent_openai = create_agent(
    model=llm_mit_compaction,
    tools=[rechner, datei_lesen],
    system_prompt=system_prompt_compaction,
    # Keine SummarizationMiddleware nötig bei OpenAI-Modellen!
)

print("✅ Agent mit OpenAI Server-Side Compaction erstellt")
print("→ compact_threshold=10000: Komprimierung ab ~10.000 Token")
print("→ Alternative zu SummarizationMiddleware(model=llm)")

# 6 | ToolRetry/ModelRetryMiddleware
---

Beide Retry-Middleware machen Agenten **robust gegen transiente Fehler**. Sie implementieren automatische Retries mit **Exponential Backoff**. In Demo-Notebooks sollte die Retry-Zahl klein bleiben, damit Fehler nicht lange hängen und keine unnötigen API-Kosten entstehen.

- `ToolRetryMiddleware` – wiederholt fehlgeschlagene **Tool-Aufrufe**
- `ModelRetryMiddleware` – wiederholt fehlgeschlagene **LLM-Aufrufe** (Rate Limits, Timeouts)

**Gemeinsame Parameter:**

| Parameter | Typ | Beschreibung |
|-----------|-----|-------------|
| `max_retries` | `int` | Maximale Anzahl Wiederholungen (Default: 2) |
| `backoff_factor` | `float` | Faktor für Exponential Backoff (z.B. 2.0 → 2s, 4s, 8s) |
| `initial_delay` | `float` | Anfängliche Wartezeit in Sekunden (Default: 1.0) |
| `max_delay` | `float` | Maximale Wartezeit (Default: 60s) |
| `jitter` | `bool` | ±25% zufällige Variation zur Lastvermeidung |
| `retry_on` | `tuple` | Exception-Typen, auf die reagiert wird |
| `on_failure` | `str` | Verhalten bei Erschöpfung: `'continue'`, `'error'`, `'return_message'` |

In [ ]:
# Agent mit Retry-Logik für Tools und Modell
system_prompt_retry = "Du bist ein hilfreicher Assistent. Antworte auf Deutsch."

agent_retry = create_agent(
    model=llm,
    tools=[rechner, datei_lesen],
    system_prompt=system_prompt_retry,
    middleware=[
        ToolRetryMiddleware(
            max_retries=3,
            backoff_factor=2.0,
            jitter=True
        ),
        ModelRetryMiddleware(
            max_retries=3,
            backoff_factor=2.0,
            jitter=True
        )
    ]
)

print("✅ Agent mit ToolRetryMiddleware + ModelRetryMiddleware erstellt")

# 7 | Middleware kombinieren
---

In [ ]:
# Production-Ready Kombination: Logging + Retry + HITL + Summarization
system_prompt_production = "Du bist ein sicherer und zuverlässiger Datei-Assistent. Antworte auf Deutsch."

agent_production = create_agent(
    model=llm,
    tools=tools,
    system_prompt=system_prompt_production,
    middleware=[
        log_before,                                    # 1. Logging: Was geht rein?
        log_after,                                     # 2. Logging: Was kam raus?
        log_tool,                                      # 3. Logging: Welches Tool?
        ModelRetryMiddleware(max_retries=3, jitter=True),  # 4. LLM-Retry bei Fehlern
        ToolRetryMiddleware(max_retries=2, jitter=True),   # 5. Tool-Retry bei Fehlern
        hitl,                                          # 6. Genehmigung für Schreiboperationen
        SummarizationMiddleware(model=llm),            # 7. Kontext-Zusammenfassung
    ],
    checkpointer=InMemorySaver()
)

print("✅ Production-Ready Agent erstellt")
print(f"   Middleware-Stack: 7 Schichten (Logging + Retry + HITL + Summarization)")

In [ ]:
# Test: Berechnung (wird geloggt, keine Genehmigung nötig)
result = agent_production.invoke(
    {"messages": [{"role": "user", "content": "Berechne 123 + 456"}]},
    config={"configurable": {"thread_id": "kombi-rechner"}}
)

print()
mprint("### 📣 Antwort:")
mprint("---")
mprint(result["messages"][-1].content)


**Best Practices:**

| Empfehlung | Beschreibung |
|------------|-------------|
| **Logging sparsam halten** | `@before_model` + `@after_model` + `@wrap_tool_call` für Transparenz, aber Inhalte redigieren oder kürzen |
| **HITL für Schreiboperationen** | Sensible Tools mit `interrupt_on` absichern |
| **Retry begrenzen** | `ModelRetryMiddleware` + `ToolRetryMiddleware` für transiente Fehler, aber mit kleinen Limits im Kurskontext |
| **Summarization für lange Sessions** | `SummarizationMiddleware` verhindert Token-Overflow |
| **Checkpointer nicht vergessen** | `HumanInTheLoopMiddleware` benötigt `checkpointer=InMemorySaver()` |

# A | Aufgabe
---

<p><font color='darkblue' size="4">
📌 <b>Wichtig</b>
</font></p>

Die Aufgabestellungen unten bieten Anregungen, Sie können aber auch gerne eine andere Herausforderung angehen.

**Hinweis zur Lösungshilfe:**
> In diesem Kurs dürfen und sollen Sie generative KI auch als Unterstützung beim Lernen und Entwickeln nutzen. Wenn Sie bei einer Aufgabe festhängen, können Sie zum Beispiel Gemini in Google Colab verwenden, um Fehlermeldungen besser zu verstehen, Ideen für Teilschritte zu bekommen oder Code-Varianten zu prüfen.
> Wichtig ist nur: Die KI dient als Lern- und Entwicklungshilfe. Der Schwerpunkt des Kurses bleibt darauf, GenAI-Apps selbst zu verstehen, aufzubauen und gezielt weiterzuentwickeln.

**Grundlagen**

Baue einen Agenten mit zwei Tools und einer einfachen `@wrap_tool_call`-Middleware, die jeden Tool-Aufruf protokolliert.

**✅ Erledigt wenn:** Das Logging zeigt für jeden Tool-Aufruf mindestens Zeitstempel und Tool-Name in der Ausgabe. Inhalte oder personenbezogene Daten werden nicht ungefiltert ausgegeben.

In [ ]:
# Grundlagen: Agent mit 2 Tools + Logging-Middleware
# Startpunkt: Agent-Beispiel aus Kapitel 2/3 als Vorlage

# 1. Zwei Tools definieren (z. B. Taschenrechner + Textverarbeitung)
# 2. Logging-Middleware: wrap_tool_call loggt Tool-Name + Zeitstempel
# 3. Agent aufrufen und Ausgabe prüfen
# ...

**Aufbau**

Integriere die drei gezeigten Middleware-Hooks (`before_model`, `after_model`, `wrap_tool_call`) und füge user-in-the-Loop für mindestens eine sensible Aktion hinzu.

**✅ Erledigt wenn:** Der Agent stoppt vor der markierten sensiblen Aktion und wartet auf Bestätigung — ohne Bestätigung wird die Aktion nicht ausgeführt.

In [ ]:
# Aufbau: Alle 3 Middleware-Hooks + user-in-the-Loop
# Startpunkt: Middleware-Beispiele aus Kapitel 3/4

# 1. before_model: Anfrage prüfen oder Logging vorbereiten
# 2. after_model: Modellentscheidung prüfen
# 3. wrap_tool_call: Tool-Aufruf loggen, messen oder Fehler behandeln
# 4. HITL: für mindestens eine sensible Aktion manuell bestätigen
# ...

**Vertiefung**

Werte Laufzeiten aus, mache die Middleware robuster gegen Ausnahmen oder ergänze weitere Guardrails (z.B. Eingabe-Validierung).

**✅ Erledigt wenn:** Die Laufzeitmessung zeigt pro Tool-Aufruf eine Zeitangabe; eine Guardrail-Verletzung erzeugt eine klare Fehlermeldung — kein stiller Fehler.

In [ ]:
# Vertiefung: Laufzeiten + Guardrails

# Option A: Laufzeitmessung
#   wrap_tool_call: time.time() Differenz pro Aufruf messen + ausgeben

# Option B: Guardrails
#   wrap_tool_call: Tool-Args auf Schlagworte prüfen (z. B. "lösche", "passwort")
#   Bei Verletzung: klare Fehlermeldung zurückgeben

# ── Stdlib ────────────────────────────────────────────────────────────────────
import time
# ...

# B | Dokumente zum Weiterlesen
---




📚 Ergänzende Artikel aus der Kurs-Dokumentation:

- [GenAI-Sicherheit](https://ralf-42.github.io/GenAI/07-qualitaet-sicherheit/genai-sicherheit.html)
- [Middleware & Integration](https://ralf-42.github.io/GenAI/10-deployment/middleware-integrationsschicht.html)
- [Minimum Viable GenAI Stack](https://ralf-42.github.io/GenAI/10-deployment/minimum-viable-genai-stack.html)